In [3]:
import torch


In [4]:
a = torch.arange(24)
b = a.reshape(2,3,4)
b

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [5]:
c = b[1:2]
c

tensor([[[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [6]:
print(f"{a.storage_offset()=}")
print(f"{c.storage_offset()=}")
print(f"{a.stride()=}")
print(f"{b.stride()=}")
print(f"{a.data_ptr()=}")
print(f"{b.data_ptr()=}")
print(f"{a.device=}")
print(f"{b.device=}")

a.storage_offset()=0
c.storage_offset()=12
a.stride()=(1,)
b.stride()=(12, 4, 1)
a.data_ptr()=4094580293952
b.data_ptr()=4094580293952
a.device=device(type='cpu')
b.device=device(type='cpu')


In [7]:
c = a.reshape(4,6)
c = c.t()

In [8]:
print(f"{a.data_ptr()=}")
print(f"{c.data_ptr()=}")
# print()

d = c.contiguous() # => returns copy since this data can't be arranged as continguous view
print(f"{d.data_ptr()=}")
print(d.stride())
print(c.stride())

a.data_ptr()=4094580293952
c.data_ptr()=4094580293952
d.data_ptr()=4094580294144
(4, 1)
(1, 6)


In [9]:
print(f"{c.is_contiguous()=}")
c.view(-1) # asking for a 1 d view , which not possible, but a reshape copies data and makes it possible

c.is_contiguous()=False


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [ ]:
e = c.reshape(-1)
print(e)
print(f"{e.data_ptr()=}")

tensor([ 0,  6, 12, 18,  1,  7, 13, 19,  2,  8, 14, 20,  3,  9, 15, 21,  4, 10,
        16, 22,  5, 11, 17, 23])
e.data_ptr()=2828638683840


In [ ]:
x = torch.arange(4)
y = x.expand((3,4)) # broadcasting (sets stride = 0 for row)
print(y)
print(f"{x.shape=}")
print(f"{y.shape=}")
print(f"{x.stride()=}")
print(f"{y.stride()=}")

tensor([[0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3]])
x.shape=torch.Size([4])
y.shape=torch.Size([3, 4])
x.stride()=(1,)
y.stride()=(0, 1)


In [ ]:
x = x.to("cuda")
#by defalut tensors have requres grad false
y = torch.ones_like(x, device= "cuda", requires_grad= True, dtype= torch.float32)
z = x  * y; z.retain_grad() # by defalut intermediate tensors don't retain grad, intermediate 
                            #intermediate tensors produced by op a tensor that has req grad true will also have req grad true
l = z.sum()
l.backward()

In [ ]:
try:
    l.backward()
except Exception as e:
    print(e)

Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.


In [ ]:
print(f"{z.grad=}")
print(f"{y.grad=}")
print(f"{x.grad=}")


z.grad=tensor([2., 2., 2., 2.], device='cuda:0')
y.grad=tensor([0., 1., 2., 3.], device='cuda:0')
x.grad=None


In [ ]:
print(f"{y.grad_fn=}")
print(f"{z.grad_fn=}")

y.grad_fn=None
z.grad_fn=<MulBackward0 object at 0x0000023A1FDCD510>


In [ ]:
#tensor with  gradfn  = none is considered a leaf tensor
print(f"{z.is_leaf=}")
print(f"{y.is_leaf=}")


z.is_leaf=False
y.is_leaf=True


In [ ]:
a = torch.tensor([1,2,3])
print(f"{a.device=}")
b = a.cuda()
print(f"{b.device=}")
print(b)

a.device=device(type='cpu')
b.device=device(type='cuda', index=0)
tensor([1, 2, 3], device='cuda:0')


In [ ]:
t = torch.arange(12).reshape(3, 4)
t.T.view(12)                 # RuntimeError: view size is not compatible...
t.T.reshape(12)              # works — reshape() falls back to a copy when needed
t.T.contiguous().view(12)    # also works — contiguous() materializes a fresh copy first

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [ ]:
import torch

In [ ]:
class Model():
    def __init__(self, nin, nout):
        self.W = torch.rand((nin, nout), requires_grad = True, device = "cuda")
        self.B = torch.rand((1, nout), requires_grad = True, device = "cuda")

    def __call__(self, data_batch, inference = False):
        if inference:
            with torch.no_grad():
                return (data_batch @ self.W) + self.B
        return (data_batch @ self.W) + self.B

    def loss(self, data_batch, y_true):
        return torch.mean(torch.square(y_true - self(data_batch))) #don't use inplace operations it will cause autograd and backprop erorrs
                                                                    #let it generate new intermediate activations
    def update(self, lr):
        with torch.no_grad():
            self.W -=  lr * self.W.grad        #in place operataion better not to record these operations (anyhow no use)
            self.B -=  lr * self.B.grad
        self.W.grad = None
        self.B.grad = None
            

In [ ]:
X =  torch.rand((50, 4)) * 100
Y =  X @ (torch.tensor([2, 4, 8, 1.0]).reshape(4,1))  + 10

In [ ]:
print(X[:10])
print(Y[:10])

tensor([[ 9.3582, 88.9302, 82.3615, 42.3378],
        [44.1508, 25.4646, 43.1591, 98.1039],
        [79.1976, 97.7842, 18.3210, 58.4280],
        [34.3602, 54.2397, 68.9917, 91.4677],
        [54.9012, 65.0315,  8.1321, 40.9615],
        [99.8326, 74.9367, 61.6253, 67.0083],
        [52.0001, 51.9766, 61.7287, 50.0969],
        [54.3866, 41.5242, 94.6772, 13.2351],
        [90.1427, 94.3073, 19.0575, 33.9468],
        [47.1536, 61.9828, 37.4498, 92.1716]])
tensor([[1085.6669],
        [ 643.5367],
        [ 764.5283],
        [ 939.0804],
        [ 485.9472],
        [1069.4227],
        [ 865.8330],
        [1055.5231],
        [ 753.9218],
        [ 744.0087]])


In [ ]:
# model(X, inference=True)[:10]
def update(model,steps, lr):
    for model in range(steps):
        loss = model.loss(X.cuda(), Y.cuda())
        if step % 1000 == 0:
            print(f"step_{step}:{loss=}")
        loss.backward()
        model.update(lr)
    print("W\n", model.W)
    print("B\n", model.B)
    #print("W:grad\n", model.W.grad)
    print(f"final loss :{model.loss(X.cuda(),Y.cuda())}")


In [ ]:
model_2 = Model(4, 1)
update(model_2, 100, 1e-2)

step_0:loss=tensor(499063.5938, device='cuda:0', grad_fn=<MeanBackward0>)
W
 tensor([[nan],
        [nan],
        [nan],
        [nan]], device='cuda:0', requires_grad=True)
B
 tensor([[nan]], device='cuda:0', requires_grad=True)
final loss :nan


: 

In [ ]:
model_1 = Model(4, 1)
update(model_1, 1000000, 1e-7)

step_0:loss=tensor(498870.9375, device='cuda:0', grad_fn=<MeanBackward0>)
step_100:loss=tensor(324213.1250, device='cuda:0', grad_fn=<MeanBackward0>)
step_200:loss=tensor(214042.3594, device='cuda:0', grad_fn=<MeanBackward0>)
step_300:loss=tensor(144419.6562, device='cuda:0', grad_fn=<MeanBackward0>)
step_400:loss=tensor(100297.5156, device='cuda:0', grad_fn=<MeanBackward0>)
step_500:loss=tensor(72217.3203, device='cuda:0', grad_fn=<MeanBackward0>)
step_600:loss=tensor(54232.9531, device='cuda:0', grad_fn=<MeanBackward0>)
step_700:loss=tensor(42606.8359, device='cuda:0', grad_fn=<MeanBackward0>)
step_800:loss=tensor(34989.0273, device='cuda:0', grad_fn=<MeanBackward0>)
step_900:loss=tensor(29902.0742, device='cuda:0', grad_fn=<MeanBackward0>)
step_1000:loss=tensor(26416.8652, device='cuda:0', grad_fn=<MeanBackward0>)
step_1100:loss=tensor(23949.0273, device='cuda:0', grad_fn=<MeanBackward0>)
step_1200:loss=tensor(22130.8242, device='cuda:0', grad_fn=<MeanBackward0>)
step_1300:loss=tens

In [ ]:
#since intial loss is very high the gradient will also be very high
#on the loss surface we are at a great height
#so we might overshoot if the learning rate is low
#so 1e-2 didn't work
#but 1e-7 worked
#this is why proper intialization, normalization of features, and learning rete schedules and optimizers are very important
#here the problem is with small intial predicted values due small weight intialization(which is normal)
#instead we should are normalized the features of the data to make them small and workable

In [ ]:
p = torch.randn((3, 4), requires_grad = True)
q = torch.randn((4,3))


In [ ]:
r = p @ q
s = r.sum()
print(f"{s=}")

s=tensor(1.3212, grad_fn=<SumBackward0>)


In [ ]:
s.backward()

In [ ]:
print(f"{p.grad=}")
print(f"{p.grad.shape=}")

p.grad=tensor([[-0.1031,  0.4161,  0.1211, -0.4388],
        [-0.1031,  0.4161,  0.1211, -0.4388],
        [-0.1031,  0.4161,  0.1211, -0.4388]])
p.grad.shape=torch.Size([3, 4])


In [ ]:
print(p.data.data_ptr())
print(p.data_ptr())
p is p.data # false because underlying storage is same but pyton tensor object is differt (view)


4053241497536
4053241497536


False

In [ ]:
n = torch.tensor([1, 2, 3], device= "cuda")
n.data  # n.data will be on same device as n(it's obivous right dumbass)


tensor([1, 2, 3], device='cuda:0')

In [36]:
k = torch.rand(2*3*4).view(2,3,4)
print(f"{k=}")

k=tensor([[[0.0445, 0.0383, 0.8053, 0.6997],
         [0.2269, 0.9906, 0.6075, 0.9202],
         [0.0623, 0.3451, 0.4620, 0.2557]],

        [[0.4958, 0.4470, 0.8147, 0.7830],
         [0.7250, 0.4715, 0.1390, 0.5169],
         [0.5584, 0.8682, 0.5464, 0.3790]]])


In [37]:
idx0 = torch.tensor([1, 0])
idx2 = torch.tensor([2, 3])
k[idx0, : , idx2]  #guess shape=(2,3), elements [[2nd table third column all row elements ], [1st table  4th column all row elements]]

tensor([[0.8147, 0.1390, 0.5464],
        [0.6997, 0.9202, 0.2557]])

In [ ]:
k[k < 0.5]

tensor([0.0445, 0.0383, 0.2269, 0.0623, 0.3451, 0.4620, 0.2557, 0.4958, 0.4470,
        0.4715, 0.1390, 0.3790])

In [ ]:
k[:,[[0,1], [1,2]],:] #shape(2,2,2,4)

tensor([[[[0.0445, 0.0383, 0.8053, 0.6997],
          [0.2269, 0.9906, 0.6075, 0.9202]],

         [[0.2269, 0.9906, 0.6075, 0.9202],
          [0.0623, 0.3451, 0.4620, 0.2557]]],


        [[[0.4958, 0.4470, 0.8147, 0.7830],
          [0.7250, 0.4715, 0.1390, 0.5169]],

         [[0.7250, 0.4715, 0.1390, 0.5169],
          [0.5584, 0.8682, 0.5464, 0.3790]]]])

In [211]:
import torch.nn as nn
dir(nn.Module)
#MODULE TAKES CARE OF BUFFERS, PARAMETERS, SUB MODULES  (PARAMETERS)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_call_impl',
 '_compiled_call_impl',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_load_from_state_dict',
 '_maybe_warn_non_full_backward_hook',
 '_named_members',
 '_register_load_state_dict_pre_hook',
 '_register_state_dict_hook',
 '_replicate_for_data_parallel',
 '_save_to_state_dict',
 '_slow_forward',
 '_version',
 '_wrapped_call_impl',
 'add_module',
 'apply',
 'bfloat16',
 'buffers',
 'call_super_init',
 'children',
 'compile',
 'cpu',


In [212]:
layer = nn.Linear(2,3)
vars(layer)

{'training': True,
 '_parameters': {'weight': Parameter containing:
  tensor([[ 0.5574,  0.4232],
          [-0.6288,  0.1098],
          [ 0.5840,  0.6834]], requires_grad=True),
  'bias': Parameter containing:
  tensor([ 0.3826, -0.3997, -0.4188], requires_grad=True)},
 '_buffers': {},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {},
 'in_features': 2,
 'out_features': 3}

In [213]:

class Network(nn.Module):
    def __init__(self, in_dim, hid_dim:list, out_dim):
        super().__init__()
        self.layers = []
        for i,j in zip([in_dim]+hid_dim, hid_dim + [out_dim]): 
            self.layers.append(nn.Linear(i,j))
            self.layers.append(nn.ReLU())
        self.layers.pop()
        self.layers = nn.ModuleList(self.layers) #moduleist is indexable and iterable

    def forward(self, inputs):  # how does __call__ know what parameters to accept maybe *args   
        for layer in self.layers:
            inputs = layer(inputs)
        return inputs

In [214]:
torch.manual_seed(1729)
X = torch.rand((5,3))
Y = (X ** 2).sum(dim = 1, keepdim= True)
print(X)
print(Y)


tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691],
        [0.2332, 0.4047, 0.2162],
        [0.9927, 0.4128, 0.5938],
        [0.6128, 0.1519, 0.0453]])
tensor([[0.3367],
        [0.1879],
        [0.2649],
        [1.5084],
        [0.4006]])


In [215]:
network = Network(3,[5,4],1)
def init_weights(module):
    if isinstance(module, nn.Linear):
        module.weight = nn.Parameter(torch.randn_like(module.weight))
network.apply(init_weights)

Network(
  (layers): ModuleList(
    (0): Linear(in_features=3, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )
)

In [216]:
print(isinstance(nn.ModuleList(), nn.Module))
network.layers[:2]

True


ModuleList(
  (0): Linear(in_features=3, out_features=5, bias=True)
  (1): ReLU()
)

In [217]:
print(network)

Network(
  (layers): ModuleList(
    (0): Linear(in_features=3, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )
)


In [218]:
print([i for i in network.parameters()])   # calling parameter funciton gives u a generator so to print it i have to do this

[Parameter containing:
tensor([[-0.4861, -0.7959, -0.5204],
        [-0.7523,  2.2930,  0.9971],
        [ 1.4798, -0.9832,  0.2359],
        [ 0.7787, -0.1732, -0.1074],
        [ 1.9456,  0.4525,  0.1014]], requires_grad=True), Parameter containing:
tensor([-0.4962,  0.0121,  0.5139, -0.3050, -0.3488], requires_grad=True), Parameter containing:
tensor([[-0.1057,  1.1269, -2.8065, -1.3972, -1.2127],
        [-1.4975, -2.0316, -1.0010, -1.5321, -0.6058],
        [-1.2782,  0.4728, -0.9737,  0.2301, -1.2427],
        [ 0.4232, -1.7144,  1.2718,  0.5791,  1.0235]], requires_grad=True), Parameter containing:
tensor([-0.4131, -0.2560,  0.0354, -0.1176], requires_grad=True), Parameter containing:
tensor([[ 0.3339, -1.4079, -0.6539, -1.3118]], requires_grad=True), Parameter containing:
tensor([0.2375], requires_grad=True)]


In [219]:
optimizer = torch.optim.SGD(network.parameters(), 0.0001) # is it right to have fit in this class and then optimizer in this function why should they be outside 
                                                            #how if optimizer is outside and if i change some weight like intialize it to deffernt intialization does
                                                            #still capture why

In [268]:
def fit(model, X, Y, optimizer, steps = 100):
        model.train()
        for step in range(steps):
            optimizer.zero_grad(True)
            y_pred  = model(X)
            # loss = torch.mean(((Y - y_pred) **2))
            # mse = nn.MSELoss()
            # loss = mse(y_pred, Y)
            loss = nn.functional.mse_loss(y_pred, Y)
            if  step % 50 == 0:
                print(f"loss at {step}: {loss}")
            loss.backward()
            optimizer.step()

In [269]:
network.cuda() #in place shits all (buffers, parameters(including submodules to gpu))
fit(network, X.cuda(), Y.cuda(), optimizer, 100000)

loss at 0: 4.087510205863509e-06
loss at 50: 4.087075922143413e-06
loss at 100: 4.086807166459039e-06
loss at 150: 4.086372427991591e-06
loss at 200: 4.086025910510216e-06
loss at 250: 4.085836280864896e-06
loss at 300: 4.085235559614375e-06
loss at 350: 4.085045929969056e-06
loss at 400: 4.084692591277417e-06
loss at 450: 4.084346073796041e-06
loss at 500: 4.0841491681931075e-06
loss at 550: 4.083881776750786e-06
loss at 600: 4.083526619069744e-06
loss at 650: 4.08302184951026e-06
loss at 700: 4.082588475284865e-06
loss at 750: 4.082233772351174e-06
loss at 800: 4.081880888406886e-06
loss at 850: 4.081611677975161e-06
loss at 900: 4.0814252315612976e-06
loss at 950: 4.080914095538901e-06
loss at 1000: 4.08071855417802e-06
loss at 1050: 4.080285634699976e-06
loss at 1100: 4.07993866247125e-06
loss at 1150: 4.079593963979278e-06
loss at 1200: 4.079313384863781e-06
loss at 1250: 4.079046902916161e-06
loss at 1300: 4.07870265917154e-06
loss at 1350: 4.078515303262975e-06
loss at 1400: 4.0

In [271]:
print(f"{Y=}")
network.eval()
with torch.no_grad():
    print(f"y_pred={network(X.cuda())}")


Y=tensor([[0.3367],
        [0.1879],
        [0.2649],
        [1.5084],
        [0.4006]])
y_pred=tensor([[0.3385],
        [0.1894],
        [0.2615],
        [1.5069],
        [0.4008]], device='cuda:0')


In [272]:
vars(network)

{'training': False,
 '_parameters': {},
 '_buffers': {'mov_avg': tensor([1, 2, 3], device='cuda:0')},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {'layers': ModuleList(
    (0): Linear(in_features=3, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )}}

In [273]:
network.buffer =  torch.tensor([1,2,3])
network.register_buffer(name= "mov_avg", tensor = network.buffer)

In [274]:
network.state_dict()

OrderedDict([('mov_avg', tensor([1, 2, 3])),
             ('layers.0.weight',
              tensor([[-0.4861, -0.7959, -0.5204],
                      [-0.6019,  2.4258,  1.1695],
                      [ 1.3415, -1.0910,  0.0991],
                      [ 0.6811, -0.2097, -0.1530],
                      [ 1.7910,  0.3113, -0.0474]], device='cuda:0')),
             ('layers.0.bias',
              tensor([-0.4962,  0.2112,  0.3165, -0.4184, -0.6633], device='cuda:0')),
             ('layers.2.weight',
              tensor([[-0.1057,  1.0641, -2.8082, -1.3972, -1.2134],
                      [-1.4975, -2.0316, -1.0010, -1.5321, -0.6058],
                      [-1.2782,  0.8971, -0.9102,  0.2301, -1.2233],
                      [ 0.4232, -1.8216,  1.1616,  0.5485,  0.8499]], device='cuda:0')),
             ('layers.2.bias',
              tensor([-0.4561, -0.2560,  0.3444, -0.1513], device='cuda:0')),
             ('layers.4.weight',
              tensor([[ 0.1522, -1.4079, -0.9796, -1.2152]

In [275]:
print(network._buffers)

{'mov_avg': tensor([1, 2, 3])}


In [276]:
network.__delattr__("buffer")  # this doesn't removie the register buffer only network.buffer obj is removed

In [277]:
print(network._buffers)

{'mov_avg': tensor([1, 2, 3])}


In [278]:
print(network)
network.state_dict()

Network(
  (layers): ModuleList(
    (0): Linear(in_features=3, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )
)


OrderedDict([('mov_avg', tensor([1, 2, 3])),
             ('layers.0.weight',
              tensor([[-0.4861, -0.7959, -0.5204],
                      [-0.6019,  2.4258,  1.1695],
                      [ 1.3415, -1.0910,  0.0991],
                      [ 0.6811, -0.2097, -0.1530],
                      [ 1.7910,  0.3113, -0.0474]], device='cuda:0')),
             ('layers.0.bias',
              tensor([-0.4962,  0.2112,  0.3165, -0.4184, -0.6633], device='cuda:0')),
             ('layers.2.weight',
              tensor([[-0.1057,  1.0641, -2.8082, -1.3972, -1.2134],
                      [-1.4975, -2.0316, -1.0010, -1.5321, -0.6058],
                      [-1.2782,  0.8971, -0.9102,  0.2301, -1.2233],
                      [ 0.4232, -1.8216,  1.1616,  0.5485,  0.8499]], device='cuda:0')),
             ('layers.2.bias',
              tensor([-0.4561, -0.2560,  0.3444, -0.1513], device='cuda:0')),
             ('layers.4.weight',
              tensor([[ 0.1522, -1.4079, -0.9796, -1.2152]

In [279]:
print(type(torch.nn))

<class 'module'>


In [ ]:
#karpathy videos
#    pytorch we will do it later again
#    what happend then
#    don't lag i just want to finish this pytorch thing
#    and code makermore 1 and 2
#    by tommorow morning ok bro
#    backend/sql/dsa
#    let's get to the videos for now
#